# INFO-6147-(01)-26W Deep Learning with Pytorch

## Project: This is for UI

**Student Name:** Yun-Jiung Wang

**Student Number:** 1256222

**Date:** March 30th, 2026

**Description:**

This project is aim to allow user upload the photos of food and describe what the food tastes like. It will be helpful when traveling or when people wants to try international food, but not having a person to explain to you what that is.
Here comes this AI project, to assist people on observing new foods.

## Setup Environment

In [ ]:
!pip install datasets gradio streamlit groq pyngrok

# dlownload Cloudflare tunnle tool
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## Create app.py for UI

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from groq import Groq
import os

# --- 1. Model Loading Configuration ---
# Use @st.cache_resource to avoid reloading the model on every interaction
@st.cache_resource
def load_food_model(model_path, num_classes=40):
    # Reconstruct the exact same ResNet-50 architecture used during training

    # model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 1024), # larger to 1024
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.4),           # prevernt overfitting
        nn.Linear(1024, 512),      # add layer
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Linear(512, num_classes)         #final classes
    )

    # Load weights to CPU or GPU automatically
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if os.path.exists(model_path):
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        print(f"✅ Model loaded successfully from {model_path}")
    else:
        print(f"⚠️ Model file not found at {model_path}. Using uninitialized weights.")

    model.to(device)
    model.eval()
    return model, device

# --- 2. Image Prediction Logic ---
def predict_dish(image, model, device, class_names):
    # Transformation must match your Training/Validation transforms
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = outputs.max(1)
        index = predicted.item()
    return class_names[index]

# --- 3. Llama AI Guide Function ---
def ask_llama_chef(food_name, api_key):
    if not api_key:
        return "❌ Please enter your Groq API Key in the sidebar!"
    try:
        client = Groq(api_key=api_key)
        # Professional prompt for a travel guide persona
        prompt = f"""
        You are a witty and expert travel guide for a Canadian tourist.
        The AI vision model has identified this dish as '{food_name}'.
        Please provide: 1. Taste & Texture, 2. A fun Travel Trivia fact, 3. Practical advice for travelers.
        Tone: Friendly and humorous. Length: Under 150 words.
        """
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error connecting to Llama: {str(e)}"

# --- 4. Streamlit UI Interface ---
st.set_page_config(page_title="Food Guide", page_icon="🍔", layout="centered")

# Sidebar for API Key and Settings
with st.sidebar:
    st.title("🛠️ Settings")
    user_api_key = st.text_input("Groq API Key", type="password")
    st.markdown("---")
    st.info("The model recognizes **40 types** of dishes.")

st.title("Food Guide")
st.write("Upload a photo of your meal, and we will identify it and tell you its story!")

# --- CONFIGURATION: Update these after your training finishes ---
# Path to the .pth file you are saving tonight
MODEL_FILE = "/data/food_best.pth"
# IMPORTANT: This list must match the EXACT order of your label_map (0 to 39)
select_food_list = [
     # --- Asia ---
    'bibimbap', 'gyoza', 'sashimi', 'pad_thai', 'pho',
    'miso_soup', 'edamame', 'spring_rolls', 'sushi', 'dumplings',
    'hummus', 'falafel', 'baklava', 'chicken_curry', 'fried_rice',

    # --- Lattino ---
    'tacos', 'enchiladas', 'guacamole', 'ceviche', 'nachos',

    # --- Classic ---
    'pizza', 'hamburger', 'hot_dog', 'steak', 'french_fries',
    'grilled_salmon', 'spaghetti_bolognese', 'lasagna', 'club_sandwich',

    # --- Hot in Social Media ---
    'tiramisu', 'cheesecake', 'macarons', 'donuts', 'waffles',
    'pancakes', 'ice_cream', 'apple_pie', 'strawberry_shortcake',

    # --- seafoods ---
    'mussels', 'oysters'
]

food_num_classes = len(select_food_list)

# --- File Uploader ---
uploaded_file = st.file_uploader("📸 Choose a food image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Display the uploaded image
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded Image', use_container_width=True)

    if st.button("Identify & Get Guide"):
        with st.spinner("Analyzing your dish..."):
            # 1. Load the trained ResNet-50 model
            model, device = load_food_model(MODEL_FILE, num_classes=food_num_classes)

            # 2. Run Inference
            predicted_label = predict_dish(image, model, device, select_food_list)
            st.success(f"Prediction: **{predicted_label.replace('_', ' ').title()}**")

            # 3. Generate Guide via Llama
            guide_result = ask_llama_chef(predicted_label, user_api_key)
            st.chat_message("assistant", avatar="👨‍🍳").write(guide_result)

## Activate Tunnel

In [ ]:
import subprocess
import time
import socket

# --- 1. Check if Streamlit is already running on port 8501 ---
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

# --- 2. Start Streamlit if it's not already running ---
if not is_port_open(8501):
    print("🚀 Starting Streamlit in the background...")
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5) # Give it time to boot
else:
    print("✅ Streamlit is already running.")

# --- 3. Start Cloudflare Tunnel and FORCE print logs ---
print("🌐 Opening Cloudflare Tunnel... (Look for the '.trycloudflare.com' link below)")
print("-" * 50)

# We use stdbuf to disable buffering so the URL appears immediately
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# This loop will print EVERYTHING Cloudflare says until the URL appears
for line in p.stdout:
    print(f"DEBUG: {line.strip()}") # This helps us see if there's an error
    if "trycloudflare.com" in line:
        url = line.strip().split(" ")[-1]
        print("\n" + "★" * 50)
        print(f"🔥 SUCCESS! YOUR APP IS LIVE AT:")
        print(f"👉 {url}")
        print("★" * 50)
        # We don't break, so the tunnel stays active in this cell